# H-001 - Kanal Konumu | GUNLUK bar

**Hipotez:** Fiyat, son N barlik kanalin dibine yakinken ileriye donuk
getirisi ortalamanin ustunde; tepesine yakinken altindadir.

**Frekans: gunluk.** Saatlik esi: `h001_hourly.ipynb`

---

## Calistirmadan once

1. `hypotheses/H-001-*.md` kaydi **yazildi ve commit'lendi** mi?
2. Bu defter ve gunluk/saatlik esi **IKI AYRI DENEME** sayilir.
   `REGISTRY.md`'de ikisi de yazili olmali (ARCHITECTURE.md 7.4).
3. Kasa (2024-07 sonrasi) bu defterde **acilmaz**.

## 0. Kararlar

In [ ]:
SYMBOL    = 'SPY'          # 'SPY' | 'AAPL'
TIMEFRAME = '1Day'

K       = 1                # 1 gun sonrasina bakiyoruz
WINDOWS = (20, 50, 100)    # ISLEM GUNU cinsinden kanal penceresi
COST    = 0.0005           # %0.05 - bunu gecmeyen hareket 'yon' sayilmaz

# Basari kriteri - sonucu GORMEDEN doldurun
N_TRIALS = len(WINDOWS) * 2   # x2: ayni hipotez saatlikte de olculuyor
ALPHA    = 0.05 / N_TRIALS
MIN_RHO  = 0.05

print(f'{N_TRIALS} deneme -> duzeltilmis esik alpha = {ALPHA:.5f}')

## 1. Veri ve bolme

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd

from src.data.load import load_bars
from lab.session import regular_hours, bars_per_day, session_report
from lab.splits import (split_research_vault, iter_subperiods,
                        iter_stress_windows, purged_walk_forward)
from lab.h001 import build_table
from lab.analysis import rank_correlation, baseline, bin_table, subperiod_report
from lab import plots

pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

In [ ]:
df = load_bars(SYMBOL, TIMEFRAME)
split = split_research_vault(df, horizon_bars=K)
split

**Kasa buradan sonra kullanilmiyor.** `split.research` ile calisiyoruz.

In [ ]:
table = build_table(split.research, windows=WINDOWS, k=K, cost=COST)
print(f'{len(table)} satir  |  {table.attrs["dropped"]} satir dustu (isinma + cevapsiz kuyruk)')
table.head()

### Ozellik neye benziyor

Sayilara gecmeden once olculen seyin ne oldugunu gormek. Ust panel kanali,
alt panel 0-1 arasi konumu gosterir.

**Grafikten 'burada alirdim' cikarimi yapmayin.** Grafige bakarken dip ve
tepe bellidir cunku sonrasini da goruyorsunuz; canli sistemde o lux yok.
Karar asagidaki rho ile verilir.

In [ ]:
plots.channel_bands(split.research, WINDOWS[0], last=250);

## 2. Baseline - hicbir sey bilmeseydik

Her sonucun yaninda bu durmali. '%54 yukari' tek basina anlamsizdir;
taban %53 ise hipotez bir sey soylemiyordur.

In [ ]:
baseline(table[f'ileri_getiri_{K}'], cost=COST)

## 3. Birincil istatistik - sira korelasyonu

Dilim/esik secimi icermez, dolayisiyla 'en iyi dilimi sec' serbestligi yoktur.

**Mean-reversion NEGATIF rho bekler:** kanal konumu arttikca (tepeye
yaklastikca) ileri getiri azalmali.

In [ ]:
rows = []
for n in WINDOWS:
    res = rank_correlation(table[f'kanal_konumu_{n}'],
                           table[f'ileri_getiri_{K}'], horizon_bars=K)
    rows.append({'N': n, 'rho': res.rho, 'p': res.p_value,
                 'n': res.n, 'n_eff': res.effective_n,
                 'gecti_mi': (res.p_value < ALPHA) and (abs(res.rho) >= MIN_RHO)})
sonuc = pd.DataFrame(rows)
sonuc

In [ ]:
plots.rho_chart([f'N={n}' for n in sonuc['N']], sonuc['rho'],
                min_rho=MIN_RHO,
                title=f'Pencereye gore sira korelasyonu - {SYMBOL} {TIMEFRAME}',
                subtitle='gri serit: ilan edilmis esigin altindaki bolge');

## 4. Dilim tablosu - yalnizca gozle gormek icin

Buradaki en iyi dilime bakip karar vermek **p-hacking**'dir. Karar
yukaridaki rho ile verildi. Bu tablo iliskinin **monotonik** olup
olmadigini gormek icin: dipten tepeye duzenli bir egim var mi?

In [ ]:
taban = baseline(table[f'ileri_getiri_{K}'], cost=COST)
dilimler = bin_table(table[f'kanal_konumu_{WINDOWS[0]}'],
                     table[f'ileri_getiri_{K}'], cost=COST)
dilimler

In [ ]:
plots.bin_chart(dilimler, reference=taban['ortalama_getiri']);

Dikey cizgiler **+/-2 standart hata**: dilim ortalamasinin belirsizligi.
Hata paylari birbirini ve tabani ortuyorsa gorunen egim gurultudur - en
parlak dilime bakip karar vermek zaten p-hacking'dir.

## 5. Alt donem tutarliligi

Etki uc rejimde de ayni isareti tasiyor mu? Tek donemde cikan etki o
rejime ozgudur - genel bir yasa degildir.

In [ ]:
donemler = subperiod_report(table,
                            feature_col=f'kanal_konumu_{WINDOWS[0]}',
                            target_col=f'ileri_getiri_{K}',
                            horizon_bars=K,
                            subperiods=iter_subperiods(table))
donemler

In [ ]:
plots.rho_chart(donemler['donem'], donemler['rho'], min_rho=MIN_RHO,
                title=f'Alt donem tutarliligi - N={WINDOWS[0]}',
                subtitle='isaret donemler arasinda degisiyorsa etki kararli degildir');

## 6. Stres penceresi raporu

**Secim kriteri degil, risk bilgisi.** Kasa kriz icermiyor; stratejinin
cokuste ne yaptigini yalnizca buradan ogrenebiliriz. Soru 'kazandi mi'
degil: **ne kadar kaybetti, dayanabilir miyiz?**

In [ ]:
for name, part in iter_stress_windows(table):
    if len(part) < 30:
        print(f'{name}: {len(part)} bar - cok az')
        continue
    b = baseline(part[f'ileri_getiri_{K}'], cost=COST)
    res = rank_correlation(part[f'kanal_konumu_{WINDOWS[0]}'],
                           part[f'ileri_getiri_{K}'], horizon_bars=K)
    print(f'{name}: {len(part):>5} bar  rho={res.rho:+.3f}  '
          f'ort_getiri={b["ortalama_getiri"]:+.5f}')

## 7. Walk-forward - etki katlar arasinda tutarli mi

Tek bir toplam sayi donemsel bir tesadufu gizleyebilir. Katlarin
cogunda ayni isaret gorunmuyorsa etki kararli degildir.

In [ ]:
rows = []
for i, (tr, te) in enumerate(purged_walk_forward(len(table), horizon_bars=K), 1):
    block = table.iloc[te]
    res = rank_correlation(block[f'kanal_konumu_{WINDOWS[0]}'],
                           block[f'ileri_getiri_{K}'], horizon_bars=K)
    span = f"{block['timestamp'].iloc[0].date()} -> {block['timestamp'].iloc[-1].date()}"
    rows.append({'kat': f'kat {i}', 'bar': len(te), 'donem': span,
                 'rho': res.rho, 'p': res.p_value})
katlar = pd.DataFrame(rows)
katlar

In [ ]:
plots.rho_chart(katlar['kat'], katlar['rho'], min_rho=MIN_RHO,
                title=f'Walk-forward katlari - N={WINDOWS[0]}',
                subtitle='katlarin cogunda ayni isaret yoksa etki kararli degildir');

---

## 8. Sonucu kaydet

Sonuc ne olursa olsun `hypotheses/H-001-*.md` dosyasina yazilir ve
`REGISTRY.md`'de deneme sayaci guncellenir. **Reddedilen hipotez
silinmez** - kac deneme yaptigimizi bilmek, kalanlarin anlamliligini
belirliyor (TD-14).

Iki defterin sonucu **yan yana** degerlendirilir: frekans karari (AK-1)
buradan cikacak.

### KASA

`split.vault` bu defterde **acilmaz**.